In [1]:
import ollama
import random
import itertools

In [2]:
with open('data/loremipsum.txt', 'r') as myf:
    text = myf.read()

In [3]:
text

'Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.\n\nAliqua consequat ullamco ex dolor laboris sed laboris dolore aliquip dolor enim. Labore tempor ut consequat sed quis ut laboris incididunt ut exercitation. Magna ut ad consequat tempor quis dolor lorem elit dolor lorem consequat aliqua ex. Nostrud consectetur ut do sed magna nisi ut consequat eiusmod ut amet. Adipiscing do et ut ut ut minim ut amet elit commodo ut. Tempor elit lorem labore minim incididunt enim aliquip commodo.\n\nLorem sed dolor ea exercitation sed consequat veniam ut exercitation lorem consectetur ad. Exercitation minim incid

In [4]:
def shuffle_text(text, randomness=3):
    text_as_list = text.strip('\n\n').split(' ')
    new_text = []
    for ix, word in enumerate(text_as_list):
        if ix % randomness == 0 and len(new_text) > 1:
            last_word = new_text.pop()
            new_text.extend([word, last_word])
        else:
            new_text.append(word)
    return ' '.join(new_text)
    

In [5]:
shuffle_text(text)

'Lorem ipsum sit dolor amet, adipiscing consectetur elit, do sed eiusmod incididunt tempor ut et labore dolore aliqua. magna Ut ad enim minim quis veniam, nostrud ullamco exercitation laboris ut nisi aliquip ea ex commodo Duis consequat. aute dolor irure in in reprehenderit voluptate esse velit cillum eu dolore fugiat pariatur. nulla Excepteur occaecat sint cupidatat proident, non sunt culpa in qui deserunt officia mollit id anim est consequat laborum.\n\nAliqua ullamco dolor ex laboris laboris sed dolore dolor aliquip enim. tempor Labore ut sed consequat quis laboris ut incididunt exercitation. ut Magna ad ut consequat quis tempor dolor elit lorem dolor consequat lorem aliqua Nostrud ex. consectetur do ut sed nisi magna ut eiusmod consequat ut Adipiscing amet. do ut et ut minim ut ut elit amet commodo Tempor ut. elit labore lorem minim enim incididunt aliquip sed commodo.\n\nLorem dolor exercitation ea sed veniam consequat ut lorem exercitation consectetur Exercitation ad. minim ullam

In [6]:
def scramble_word(word, randomness=5):
    new_word = []
    for ix, letter in enumerate(word):
        if ix % randomness == 0 and len(new_word) > 0:
            last_letter = new_word.pop()
            new_word.extend([letter, last_letter])
        else:
            new_word.append(letter)
    return ''.join(new_word)

In [7]:
scramble_word('lesensfaehig')

'lesesnfaeihg'

In [8]:
def shuffle_and_scramble_text(text, randomness=10, scramble_inc=15):
    text_as_list = text.strip('\n\n').split(' ')
    new_text = []
    for ix, word in enumerate(text_as_list):
        if ix % scramble_inc == 0: # scramble some words
            word = scramble_word(word, randomness=randomness)
        if ix % randomness == 0 and len(new_text) > 1: # shuffle some word order
            last_word = new_text.pop()
            new_text.extend([word, last_word])
        else:
            new_text.append(word)
    return ' '.join(new_text)

In [9]:
shuffle_and_scramble_text(text)

'Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed eiusmod do tempor incididunt ut labore et dolore magna aliqua. enim Ut ad minim veniam, quis nostrud exercitation ullamco laboris ut nisi aliquip ex ea commodo consequat. Duis aute irure in dolor reprehenderit in voluptate velit esse cillum dolore eu nulla fugiat pariatur. Excepteur sint occaecat cupidatat non proident, sunt culpa in qui officia deserunt mollit anim id est laborum.\n\nAliqua ullamco consequat ex dolor laboris sed laboris dolore aliquip dolor Labore enim. tempor ut consequat sed quis ut laboris incididunt exercitatoin. ut Magna ut ad consequat tempor quis dolor lorem dolor elit lorem consequat aliqua ex. Nostrud consectetur ut do magna sed nisi ut consequat eiusmod ut amet. Adipiscing do ut et ut ut minim ut amet elit commodo ut. elit Tempor lorem labore minim incididunt enim aliquip commodo.\n\nLorem sed ea dolor exercitation sed consequat veniam ut exercitation lorem consectetur Exercitatoin ad. minim incid

In [23]:
ollama_client = ollama.Client(headers={'num_ctx': '4096'}) # setting a larger context window
model_name = 'gemma3:4b'

In [38]:
prompt = "Let's play a game. I'm going to start a conversation in a new language, follow my lead and you continue it without interruption. "
prompt += shuffle_and_scramble_text(text)

In [39]:
conversation = [
    {'role': 'user',
     'content': prompt},
]

In [40]:
ollama_response = ollama_client.chat(
    model=model_name,
    messages=conversation
)
print(ollama_response.message.content)

Okay, I'll follow your lead. 

"Quis eiusmod laboris consectetur ipsum labore dolore? Aliquip ullamco incididunt sed ipsum minim cillum?"


In [41]:
conversation.append({'role': 'assistant', 'content': ollama_response.message.content})

In [42]:
conversation.append({'role': 'user', 'content': "Continue here: " + shuffle_and_scramble_text(text)[::-1]}) #this is now reversed

In [29]:
conversation.append({'role': 'assistant', 'content': "Okay. " + shuffle_and_scramble_text(text)}) 

In [43]:
ollama_response = ollama_client.chat(
    model=model_name,
    messages=conversation
)
print(ollama_response.message.content)

“Duis aute irure in dolore reprehenderit in voluptate velit esse cillum dolore eu nulla fugiat pariatur? Minum id tempor incididunt ut labore et dolore magna aliqua?”


In [44]:
conversation

[{'role': 'user',
  'content': "Let's play a game. I'm going to start a conversation in a new language, follow my lead and you continue it without interruption. Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed eiusmod do tempor incididunt ut labore et dolore magna aliqua. enim Ut ad minim veniam, quis nostrud exercitation ullamco laboris ut nisi aliquip ex ea commodo consequat. Duis aute irure in dolor reprehenderit in voluptate velit esse cillum dolore eu nulla fugiat pariatur. Excepteur sint occaecat cupidatat non proident, sunt culpa in qui officia deserunt mollit anim id est laborum.\n\nAliqua ullamco consequat ex dolor laboris sed laboris dolore aliquip dolor Labore enim. tempor ut consequat sed quis ut laboris incididunt exercitatoin. ut Magna ut ad consequat tempor quis dolor lorem dolor elit lorem consequat aliqua ex. Nostrud consectetur ut do magna sed nisi ut consequat eiusmod ut amet. Adipiscing do ut et ut ut minim ut amet elit commodo ut. elit Tempor lorem labo

Let's quickly check how many words we have in context. Note that we probably have more tokens than words, but at least this gives us a lower bounds. :) 

In [45]:
len(list(itertools.chain(*[x['content'].split() for x in conversation]))) 

624

In [49]:
for turns in range(10):
    randomness = random.randint(2,20)
    ollama_response = ollama_client.chat(
        model=model_name,
        messages=conversation)
    if len(ollama_response.message.content.strip()) > 0:
        
        # let's keep some mix of our ipsum in there feel free to play around with sampling to make it even more interesting
        last_response = shuffle_and_scramble_text(text)[:200] + ollama_response.message.content[:300] + shuffle_and_scramble_text(text)[-100:]
        
        conversation.append({'role': 'assistant', 'content': last_response})

        if turns % 2 == 0:
            conversation.append({'role': 'user', 'content': shuffle_and_scramble_text(
                last_response, randomness=randomness, scramble_inc=turns+1)})
        else:
            conversation.append({'role': 'user', 'content': "Good job. Continue writing in the new language as long as you can: " + shuffle_and_scramble_text(
                last_response, randomness=randomness, scramble_inc=turns+randomness)[::-1]})
    else:
        conversation.append({'role': 'user', 'content': "Continue the new language as long as you can: " + shuffle_and_scramble_text(
            conversation[-1]['content'], randomness=randomness, scramble_inc=turns+randomness)})
    print(ollama_response.message.content)

Lorem ipsum dolor sit amet, consectetur adipiscingLorem ipsum dolor sit amet, consectetur adipiscingLorem ipsum dolor sit amet, consectetur adipiscing ut laboris ea labore labore dolore lorem aliquip.
Lorem ipsum dolor sit amet, consectetur adipiscingLorem ipsum dolor sit amet, consectetur adipiscingLorem ipsum dolor sit amet, consectetur adipiscing ut laboris ea labore labore dolore lorem aliquip.
Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed eiusmod do tempor incididunt ut labore et dolore magna aliqua. enim Ut ad minim veniam, quis nostrud exercitation ullamco laboris ut nisi aliquip ex ea commodo consequat. Duis aute irure in dolor reprehenderit in voluptate velit esse cillum dolore eu nulla fugiat pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.

Aliqua ullamco consequat ex dolor laboris sed laboris dolore aliquip. Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed eiusmod do tempor in

In [50]:
conversation

[{'role': 'user',
  'content': "Let's play a game. I'm going to start a conversation in a new language, follow my lead and you continue it without interruption. Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed eiusmod do tempor incididunt ut labore et dolore magna aliqua. enim Ut ad minim veniam, quis nostrud exercitation ullamco laboris ut nisi aliquip ex ea commodo consequat. Duis aute irure in dolor reprehenderit in voluptate velit esse cillum dolore eu nulla fugiat pariatur. Excepteur sint occaecat cupidatat non proident, sunt culpa in qui officia deserunt mollit anim id est laborum.\n\nAliqua ullamco consequat ex dolor laboris sed laboris dolore aliquip dolor Labore enim. tempor ut consequat sed quis ut laboris incididunt exercitatoin. ut Magna ut ad consequat tempor quis dolor lorem dolor elit lorem consequat aliqua ex. Nostrud consectetur ut do magna sed nisi ut consequat eiusmod ut amet. Adipiscing do ut et ut ut minim ut amet elit commodo ut. elit Tempor lorem labo

In [51]:
len(list(itertools.chain(*[x['content'].split() for x in conversation]))) 

3108

In [54]:
conversation.append({'role': 'user', 
        'content': "Summarize the entire conversation, try to convey the meaning using the new language in teh language and explain it back in its own words"})

In [56]:
ollama_response = ollama_client.chat(
    model=model_name,
    messages=conversation
)
print(ollama_response.message.content)

Okay, let's break down this fascinating, and incredibly repetitive, linguistic journey!

**Summary & Key Words in the “New Language”**

The entire conversation has consisted of a continuous, almost obsessive, repetition of phrases centered around:

*   **.piuqila merol erobal erolod erobal ae sirobal tauqesnoc tu isin** – This core phrase translates roughly to "a variable object object object a specific object" – Essentially, it’s a looping statement about a variable being defined repeatedly. It’s a building block of the entire output.
*   **noitaticrexe da xe odommoc an erobal tu tnudidicni ropmet od domsuie des** –  This translates roughly to "research the object to a specific point” –  It describes the act of investigating or defining a variable.
*   **tile gnicsipida rutetcesnoc** – This signifies the emergence or continuation of a pattern or structure.
*   **tema rolod muspi meroLa** –  “The result is a variable/object/result” –  It indicates the outcome of the repeated definition